In [0]:
%sql
-- =====================================================
-- Exploration des System Tables disponibles
-- =====================================================

SHOW TABLES IN system.lakeflow;

In [0]:
%sql
-- =====================================================
-- Monitoring : Historique des exécutions du pipeline
-- =====================================================

SELECT 
    job_id,
    run_id,
    result_state,
    period_start_time,
    period_end_time,
    ROUND(
        (unix_timestamp(period_end_time) - unix_timestamp(period_start_time)) / 60.0, 
        2
    ) AS duration_minutes
FROM system.lakeflow.job_run_timeline
WHERE job_id IN (
    SELECT job_id FROM system.lakeflow.jobs 
    WHERE name = 'banking_lakehouse_pipeline'
)
ORDER BY period_start_time DESC
LIMIT 20;

In [0]:
%sql
-- =====================================================
-- Monitoring : Performance par tâche (identifier les lenteurs)
-- =====================================================

SELECT 
    task_key,
    result_state,
    COUNT(*) AS nb_executions,
    ROUND(AVG((unix_timestamp(period_end_time) - unix_timestamp(period_start_time)) / 60.0), 2) AS avg_duration_minutes,
    ROUND(MAX((unix_timestamp(period_end_time) - unix_timestamp(period_start_time)) / 60.0), 2) AS max_duration_minutes
FROM system.lakeflow.job_task_run_timeline
WHERE job_id IN (
    SELECT job_id FROM system.lakeflow.jobs 
    WHERE name = 'banking_lakehouse_pipeline'
)
GROUP BY task_key, result_state
ORDER BY avg_duration_minutes DESC;

In [0]:
%sql
-- =====================================================
-- Monitoring : Taux de succès global (SLA)
-- =====================================================

SELECT 
    COUNT(*) AS total_runs,
    SUM(CASE WHEN result_state = 'SUCCEEDED' THEN 1 ELSE 0 END) AS runs_succeeded,
    SUM(CASE WHEN result_state = 'CANCELLED' THEN 1 ELSE 0 END) AS runs_cancelled,
    SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) AS runs_failed,
    ROUND(
        100.0 * SUM(CASE WHEN result_state = 'SUCCEEDED' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS taux_succes_pct
FROM system.lakeflow.job_run_timeline
WHERE job_id IN (
    SELECT job_id FROM system.lakeflow.jobs 
    WHERE name = 'banking_lakehouse_pipeline'
);

In [0]:
%sql
-- =====================================================
-- Data Quality Monitoring : Suivi du taux de validité
-- =====================================================

SELECT 
    'silver.clients' AS table_name,
    COUNT(*) AS total_lignes,
    SUM(CASE WHEN dq_valid_creditscore THEN 1 ELSE 0 END) AS creditscore_valides,
    SUM(CASE WHEN dq_valid_age THEN 1 ELSE 0 END) AS age_valides,
    SUM(CASE WHEN dq_valid_balance THEN 1 ELSE 0 END) AS balance_valides,
    SUM(CASE WHEN dq_valid_geography THEN 1 ELSE 0 END) AS geography_valides,
    ROUND(100.0 * SUM(CASE WHEN dq_is_valid THEN 1 ELSE 0 END) / COUNT(*), 2) AS taux_qualite_global_pct
FROM banking_lakehouse.silver.clients;

In [0]:
%sql
-- =====================================================
-- Data Quality Monitoring : Transactions
-- =====================================================

SELECT 
    'silver.transactions' AS table_name,
    COUNT(*) AS total_lignes,
    SUM(CASE WHEN dq_valid_amount THEN 1 ELSE 0 END) AS amount_valides,
    SUM(CASE WHEN dq_valid_time THEN 1 ELSE 0 END) AS time_valides,
    SUM(CASE WHEN dq_valid_class THEN 1 ELSE 0 END) AS class_valides,
    ROUND(100.0 * SUM(CASE WHEN dq_is_valid THEN 1 ELSE 0 END) / COUNT(*), 2) AS taux_qualite_global_pct
FROM banking_lakehouse.silver.transactions;

In [0]:
%sql
-- Diagnostic de confirmation avant correction
SELECT transaction_id, COUNT(*) as nb_occurrences
FROM banking_lakehouse.silver.transactions
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY nb_occurrences DESC
LIMIT 5;